---

digital brain MEG pipeline

---

In [2]:
# autoload
%load_ext autoreload
%autoreload 2

# load pgl actions
from pgl import pglActions, pglActionsMNE, pglStorage, pgl
pgl = pgl()

================================ pglBase: init =================================
(pgl) mglMetal error log can be viewed in MacOS Console app by searching for PROCESS mglMetal or in a terminal with:
      log stream --level info --process mglMetal
(pgl) To search for something specifc, e.g. messages from mglMovie:
      log stream --predicate 'eventMessage CONTAINS "mglMovie"' --style syslog --level info
(pgl->pglBase:checkOS) Python version: 3.12.14 | packaged by conda-forge | (main, Aug 21 2026, 22:39:36) [Clang 19.1.7 ]
(pgl->pglBase:checkOS) Running on MacBook Pro (MacBookPro18,3) with macOS version: 26.6.2
(pgl->pglBase:checkOS) Apple M1 Pro Cores: 8 (6 Performance and 2 Efficiency) Memory: 32 GB
(pgl->pglBase:checkOS) GPU: Apple M1 Pro (Built-In) 14 cores, Metal 4 support
(pgl->pglBase:checkOS)    Color LCD [Main Display]: 3024 x 1964 Retina (Built-in Liquid Retina XDR Display) GammaTable size: 1024
(pgl->pglBase:__init__) Main library instance created


In [3]:
pgl.settings()

(pglSettings:save) Saved settings to /Users/justin/.pgl/settings/window.json


In [ ]:
import errno
import os
import shutil
import uuid
from pathlib import Path
from tempfile import TemporaryDirectory
from unittest.mock import patch

import fsspec
import mne
import numpy as np

from pgl.pglSession import pglMNEIO, pglMNEBundleBacking
from pgl.pglSession import pglMNECache


def mustRaise(exceptionType, function):
    try: function()
    except exceptionType: return
    raise AssertionError(f"Expected {exceptionType.__name__}")


class TestSource:
    protocol = "memory"

    def __init__(self, filesystem, sessionID):
        self.filesystem = filesystem
        self.mneCacheContext = {"namespace": "testdb-testschema-server", "sessionID": sessionID}
        self.reads = []

    def open(self, path, mode="rb", **kwargs):
        self.reads.append(path)
        return self.filesystem.open(path, mode=mode, **kwargs)


print(f"MNE: {mne.__version__}; NumPy: {np.__version__}")

with TemporaryDirectory(prefix="pgl-session-cache-test-") as temporary:
    root, readers = Path(temporary).resolve(), []
    memoryFS, memoryRoot = fsspec.filesystem("memory"), f"/pgl-cache-test-{uuid.uuid4().hex}"

    try:
        rng = np.random.default_rng(37)
        info = mne.create_info(["EEG001", "EEG002"], 100.0, "eeg")
        samples = rng.normal(scale=1e-6, size=(2, 180000))
        raw = mne.io.RawArray(samples, info, verbose=False)
        bundle = pglMNEIO.saveRaw(raw, f"{memoryRoot}/r000000", filesystem=memoryFS, splitSize="2MB", bufferSizeSec=1.0)
        assert len(bundle["files"]) > 1, "Test did not create split FIF."

        cache = root / "cache"
        source1, source2 = TestSource(memoryFS, 101), TestSource(memoryFS, 102)
        backing1 = pglMNEBundleBacking(source1, bundle, cache)
        backing2 = pglMNEBundleBacking(source2, bundle, cache)

        assert not cache.exists()
        assert not source1.reads and not source2.reads
        print("PASS: backing construction performs no cache or payload I/O")

        first = pglMNEIO.loadRaw(backing1)
        readers.append(first)
        entry1 = Path(backing1._localEntrypoint)
        namespaceRoot = cache / source1.mneCacheContext["namespace"]

        assert first.preload is False
        assert backing1._localStrategy == "materialized"
        assert len(source1.reads) == len(bundle["files"])
        assert entry1 == namespaceRoot / "sessions/session000101/mne/r000000" / bundle["entrypoint"]
        assert not (namespaceRoot / "objects").exists()
        np.testing.assert_allclose(first.get_data(), samples, rtol=1e-12, atol=1e-14)
        print("PASS: complete split bundle cached directly under the session resource")

        second = pglMNEIO.loadRaw(backing2)
        readers.append(second)
        entry2 = Path(backing2._localEntrypoint)

        assert not source2.reads, "Second session reread original payloads."
        assert second.preload is False
        assert entry2 == namespaceRoot / "sessions/session000102/mne/r000000" / bundle["entrypoint"]
        np.testing.assert_allclose(second.get_data(), samples, rtol=1e-12, atol=1e-14)

        # Determine whether this actual filesystem supports hard links.
        probeSource, probeTarget = root / "probe-source", root / "probe-target"
        probeSource.write_bytes(b"probe")

        try:
            os.link(probeSource, probeTarget)
            linksSupported = True
        except OSError:
            linksSupported = False

        for member in bundle["files"]:
            alias1, alias2 = entry1.parent / member["name"], entry2.parent / member["name"]
            pglMNECache._verify(alias1, member)
            pglMNECache._verify(alias2, member)
            if linksSupported: assert os.path.samefile(alias1, alias2)

        print(f"PASS: cross-session verified reuse; hard links supported={linksSupported}")

        # The same session/resource reuses its published directory.
        repeatedSource = TestSource(memoryFS, 102)
        repeatedBacking = pglMNEBundleBacking(repeatedSource, bundle, cache)
        repeated = pglMNEIO.loadRaw(repeatedBacking)
        readers.append(repeated)

        assert repeatedBacking._localEntrypoint == backing2._localEntrypoint
        assert not repeatedSource.reads
        print("PASS: repeated restoration reuses the same verified session bundle")

        # Force a verified-copy fallback.
        source3 = TestSource(memoryFS, 103)
        backing3 = pglMNEBundleBacking(source3, bundle, cache)

        with patch.object(os, "link", side_effect=OSError(errno.EXDEV, "Simulated unsupported hard link")):
            third = pglMNEIO.loadRaw(backing3)

        readers.append(third)
        entry3 = Path(backing3._localEntrypoint)
        assert not source3.reads

        for member in bundle["files"]:
            alias1, alias3 = entry1.parent / member["name"], entry3.parent / member["name"]
            assert not os.path.samefile(alias1, alias3)
            pglMNECache._verify(alias3, member)

        np.testing.assert_allclose(third.get_data(), samples, rtol=1e-12, atol=1e-14)
        print("PASS: unsupported hard links fall back to verified copies")

        # Close session 101's reader before removing its cache directory.
        first.close()
        readers = [reader for reader in readers if reader is not first]
        shutil.rmtree(namespaceRoot / "sessions/session000101")

        assert entry2.is_file()
        np.testing.assert_allclose(second.get_data(), samples, rtol=1e-12, atol=1e-14)
        print("PASS: deleting one inactive session cache preserves the other session")

        # Close all readers before intentional cache corruption.
        for reader in readers: reader.close()
        readers.clear()

        member = bundle["files"][0]
        damaged = entry2.parent / member["name"]

        with damaged.open("r+b") as stream:
            stream.seek(-1, 2)
            original = stream.read(1)
            stream.seek(-1, 2)
            stream.write(bytes([original[0] ^ 1]))

        source4 = TestSource(memoryFS, 104)
        backing4 = pglMNEBundleBacking(source4, bundle, cache)

        with patch.object(mne.io, "read_raw_fif", side_effect=AssertionError("Reader opened corrupt cached bytes")):
            mustRaise(ValueError, lambda: pglMNEIO.loadRaw(backing4))

        assert not source4.reads
        assert backing4._localEntrypoint is None
        assert not list(namespaceRoot.rglob(".building-*"))
        print("PASS: cache corruption fails before reader initialization")

        # Remove all test session caches: no hidden canonical FIF remains.
        shutil.rmtree(namespaceRoot / "sessions")
        assert not list(namespaceRoot.rglob("*.fif"))
        assert not (namespaceRoot / "objects").exists()
        assert (namespaceRoot / ".cache.lock").is_file()
        print("PASS: deleting all session caches leaves no cached FIF payloads")

    finally:
        for reader in readers: reader.close()
        if memoryFS.exists(memoryRoot): memoryFS.rm(memoryRoot, recursive=True)

print("All session-owned hard-link cache tests passed.")

MNE: 1.12.1; NumPy: 2.5.2
PASS: backing construction performs no cache or payload I/O
PASS: complete split bundle cached directly under the session resource
PASS: cross-session verified reuse; hard links supported=True
PASS: repeated restoration reuses the same verified session bundle


AttributeError: module 'pgl' has no attribute 'pglMNECache'

In [3]:
import inspect
from pathlib import Path
from pgl.pglSession import pglMNEIO

expected = Path(cache).resolve() / "testdb-testschema-server/sessions/session000101/mne/r000000" / bundle["entrypoint"]

print(f"Actual entrypoint:   {backing1._localEntrypoint}")
print(f"Expected entrypoint: {expected}")
print(f"Strategy:           {getattr(backing1, '_localStrategy', None)!r}")
print(f"preload:            {first.preload!r}")
print(f"Source reads:       {len(source1.reads)}; expected {len(bundle['files'])}")
print("\nActive materializer:\n" + inspect.getsource(pglMNEIO._materializeFifBundle))

Actual entrypoint:   /private/var/folders/n2/4bqdpl1s6wd9x4l9ll4p03_m0000gq/T/pgl-session-cache-test-gbrqcoh2/cache/pgl-fif-ik9bhrxu/data_raw.fif
Expected entrypoint: /private/var/folders/n2/4bqdpl1s6wd9x4l9ll4p03_m0000gq/T/pgl-session-cache-test-gbrqcoh2/cache/testdb-testschema-server/sessions/session000101/mne/r000000/data_raw.fif
Strategy:           'materialized'
preload:            False
Source reads:       3; expected 3

Active materializer:
    @classmethod
    def _materializeFifBundle(cls, backing):
        """Copy and verify a complete FIF bundle into caller-owned storage.

        Failed transfers are removed. Successfully verified materializations
        remain caller-owned, including when a subsequent MNE reader fails.

        Does not cache an entrypoint before successful reader initialization.
        """
        import shutil
        from pathlib import Path
        from tempfile import mkdtemp

        bundle = cls._validateFifBundle(backing)

        cacheRoot = Pat

In [2]:
from pathlib import Path
from unittest.mock import patch
import uuid
import numpy as np

from pgl.pglSession import pglSession, pglMNEIO

checkpointID = 47
testRoot = Path("/Users/justin/pgl-mne-roundtrip-tests/aa42c90c24a048f985cdcf47b1e13044")
payloadRoot = testRoot / "payloads"
cache = testRoot / f"diagnostic-cache-{uuid.uuid4().hex}"
lines, diagnosticRaw = [], None

try:
    diagnosticSession = pglSession.loadSession(checkpointID, settingsName=globals().get("SETTINGS_NAME"), sslRootCert=globals().get("SSL_ROOT_CERT"), cacheDirectory=cache, directLocalPolicy="immutable")
    if diagnosticSession is None: raise RuntimeError("Loading checkpoint returned None.")

    rawBacking = diagnosticSession.mne._lazyRuntime().pending["raws"].backings[0]
    epochsBacking = diagnosticSession.mne._lazyRuntime().pending["epochs"]

    with patch.object(pglMNEIO, "_materializeFifBundle", side_effect=AssertionError("Eligible local FIF was copied")), patch.object(pglMNEIO, "_verifyLocalFifFiles", wraps=pglMNEIO._verifyLocalFifFiles) as verified:
        diagnosticRaw = diagnosticSession.mne.getRawAt(0)
        diagnosticEpochs = diagnosticSession.mne.epochs

        lines.append(f"Verification calls after initial access: {verified.call_count} (expected 2)")

        for name, backing, reader in (("Raw", rawBacking, diagnosticRaw), ("Epochs", epochsBacking, diagnosticEpochs)):
            lines.append(f"{name}: strategy={backing._localStrategy!r}, preload={reader.preload!r}")
            lines.append(f"  Physical entrypoint: {backing._localEntrypoint}")
            lines.append(f"  Inside this test's payload root: {Path(backing._localEntrypoint).is_relative_to(payloadRoot)}")

        checks = {
            "Repeated Raw identity": diagnosticSession.mne.getRawAt(0) is diagnosticRaw,
            "Duplicate Raw identity": diagnosticSession.mne.getRawAt(1) is diagnosticRaw,
            "Current Raw alias identity": diagnosticSession.mne.raw is diagnosticRaw,
            "Repeated Epochs identity": diagnosticSession.mne.epochs is diagnosticEpochs,
        }
        lines.extend(f"{name}: {passed}" for name, passed in checks.items())
        lines.append(f"Verification calls after repeated access: {verified.call_count} (expected 2)")

        expectedSamples = np.random.default_rng(29).normal(scale=1e-6, size=(2, 2000))
        expectedEpochs = np.stack([expectedSamples[:, start:start + 21] for start in (200, 600, 1000)])
        np.testing.assert_allclose(diagnosticRaw.get_data(), expectedSamples, rtol=1e-12, atol=1e-14)
        np.testing.assert_allclose(diagnosticEpochs.get_data(), expectedEpochs, rtol=1e-12, atol=1e-14)
        lines.append("Raw/Epochs sample comparison: PASS")

    lines.append(f"Cache created: {cache.exists()} (expected False)")

finally:
    if diagnosticRaw is not None: diagnosticRaw.close()
    print("\n".join(lines))

(pglStorage:loadSession) Loaded derived checkpoint 47; backed fields will load on demand.
Loading data for 3 events and 21 original time points ...
Verification calls after initial access: 2 (expected 2)
Raw: strategy='direct-local', preload=False
  Physical entrypoint: /Users/justin/pgl-mne-roundtrip-tests/94988d2627054a4db60bc41638e470d4/payloads/session000044/mne/objects/r000000/data_raw.fif
  Inside this test's payload root: False
Epochs: strategy='direct-local', preload=False
  Physical entrypoint: /Users/justin/pgl-mne-roundtrip-tests/94988d2627054a4db60bc41638e470d4/payloads/session000044/mne/objects/r000001/data-epo.fif
  Inside this test's payload root: False
Repeated Raw identity: True
Duplicate Raw identity: True
Current Raw alias identity: True
Repeated Epochs identity: True
Verification calls after repeated access: 2 (expected 2)
Raw/Epochs sample comparison: PASS
Cache created: False (expected False)


---

Help on all actions available

---

In [ ]:
pglActionsMNE.help()
#pglActionsMNE.help("loadFieldline")

---

load session

---

In [ ]:

# Load the session
#loadSession = pglActions.loadSession()
#loadSession.configure()
#if loadSession: session = loadSession.run()

---

Load FieldLine data

---

In [ ]:
# execute load
session = pglActionsMNE.loadFieldline.execute()

---

Concatenate raws

---

In [ ]:
# configure the concatenation
newSession = pglActionsMNE.concatenate.execute(session=session, all=True)
if newSession: session = newSession

---

Filter data

---

In [ ]:
session = pglActionsMNE.filter.execute(session=session, lowCutoff=0.5, highCutoff=50.0, notch=True, notchFrequency=60.0)
if newSession: session = newSession

---

Bads handling

---

In [ ]:
session = pglActionsMNE.bads.execute(session=session, excludeWhenPositionIsNaN=True, extraBads=[], interpolationOrigin="zero", method="interpolate")
if newSession: session = newSession

---

Configure events (for epoching), sets names of conditions

---

In [ ]:
# group all the things labels into one
grouped = {
    'things': range(2,201),
    'blank': 1022,
    'catch': 1023
}

# label each stimulus with its own name
each = {
    f"stimulus {code:03d}": code
    for code in range(2, 201)
}
each["blank"] = 1022
each["catch"] = 1023

triggerLabelSets = {
    "grouped": grouped,
    "each": each
}

# configure the events
newSession = pglActionsMNE.configureEvents.execute(session=session, triggerChannel='di2', triggerShortestEvent=1, triggerLabelSets = triggerLabelSets)
if newSession: session = newSession

---

create epochs

---

In [ ]:
newSession = pglActionsMNE.createEpochs.execute(session=session, tmin=0.0, tmax=2.0)
if newSession: session = newSession

---

Mark interruns as bad

---

In [ ]:
newSession = pglActions.mneMarkInterRunsAsBad.execute(session=session, tmin=0.0, tmax=2.0)
if newSession: session = newSession

---

Plot evoked

---

In [ ]:
mnePlotEvoked = pglActions.mnePlotEvoked()
#mnePlotEvoked.configure(set="grouped",label="things",minFreq=0,maxFreq=20,picks="L213")
#mnePlotEvoked.configure(set="each",label="stimulus 010")
mnePlotEvoked.configure(set="grouped",label="things",minFreq=0,maxFreq=20)

In [ ]:
newSession = mnePlotEvoked.run(session)
if newSession: session = newSession

---

Eyeblink rejection

---

In [ ]:
# fit and display ICA components
newSession = pglActions.mneICA.execute(session=session, chType='mag',nComponents=20)
if newSession: session = newSession

In [ ]:
# mark as bad based on ICA component from above and threshold in std
newSession = pglActions.mneMarkBlinks.execute(session=session, component=8, threshold=2)
if newSession: session = newSession

In [ ]:
# Create epochs, respecting the bads from the eyeblink (will fill time periods with NaN)
newSession = pglActions.mneCreateEpochsWithBads.execute(session=session)
if newSession: session = newSession

---

Plot evoked respecting Nans introduced by blink detection

---

In [ ]:
mnePlotEvokedNan = pglActions.mnePlotEvokedNan()
#mnePlotEvoked.configure(set="grouped",label="things",minFreq=0,maxFreq=20,picks="L213")
#mnePlotEvoked.configure(set="each",label="stimulus 010",picks='eeg')
#mnePlotEvoked.configure(set="grouped",label="catch",minFreq=0,maxFreq=20,picks='eeg')
mnePlotEvokedNan.configure(set="grouped",label="things",minFreq=0,maxFreq=20,picks='mag')
#mnePlotEvoked.configure(set="each",label="stimulus 021",minFreq=0,maxFreq=20,picks='eeg')

In [ ]:
newSession = mnePlotEvokedNan.run(session=session)

---

Plot evoked each NaN version

---

In [ ]:
newSession = pglActions.mnePlotEvokedEachNan.execute(session=session, set="each", picks='mag',topoMin =-1500, topoMax=1500, timeseriesMin=-1500, timeseriesMax=1500, topoTime=284)
if newSession: session = newSession

---

Downsample epochs

---

In [ ]:
mneDownsampleEvoked = pglActions.mneDownsampleEvoked()
mneDownsampleEvoked.configure(downsampleFrequency=200)

In [ ]:
newSession = mneDownsampleEvoked.run(session)
if newSession: session = newSession

---

Compute NCSNR

---

In [ ]:
mneComputeNCSNR = pglActions.mneComputeNCSNR()
#mneComputeNCSNR.configure(set="grouped",label="things",conditionIDColumn="code",chType="mag",channelName="L213_bz-s115")
#mneComputeNCSNR.configure(set="grouped",label="things",conditionIDColumn="code",chType="mag")
mneComputeNCSNR.configure(set="grouped",label="things",conditionIDColumn="code",chType="mag",channelName="L213_bz-s115")


In [ ]:
newSession = mneComputeNCSNR.run(session)
if newSession: session = newSession

---

look at spectrum

---

---

History

---

In [ ]:
session.history()

---

SCRATCH code

---

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as pathEffects

channelName, peakTime, peakAmplitude = session.mne.evoked.get_peak(ch_type="mag", mode="abs", return_amplitude=True)
peakTime=0.250
with plt.rc_context({"figure.constrained_layout.use": False, "figure.autolayout": False}):
    fig = session.mne.evoked.plot_topomap(
        times=peakTime,
        ch_type="mag",
        average=0.02,
        sensors=True,
        show_names=True,
        show=False,
    )

    fig.set_size_inches(24, 16, forward=True)

    # ax.texts contains the channel-name labels added to the topomap.
    for ax in fig.axes:
        for text in ax.texts:
            text.set_color("white")
            text.set_fontsize(12)
            text.set_fontweight("bold")
            text.set_path_effects([
                pathEffects.Stroke(linewidth=2.5, foreground="black"),
                pathEffects.Normal(),
            ])

    fig.suptitle(f"Topomap at peak: {channelName} | {peakTime * 1000:.1f} ms", fontsize=18, fontweight="bold")
    plt.show()
print(session.mne.evoked.ch_names)

In [ ]:
session.mne.lookupSensor('L101')

Do spectrum on the evoked averages - take the mean of the 3000 trials in time domain and transform last
do it over 2 seconds - you should see nothing at 0.5Hz 

Compute how many seconds (in averages) and how many trials
Same yaxis
to look at sqrt(n) - compute 
Deflate the noise to compare to the number of trials

In [ ]:
%matplotlib qt
import mne
mne.viz.set_browser_backend("qt")
#session.mne.raw.plot(duration=20, n_channels=30, scalings="auto", block=True)
session.mne.raw.compute_psd().plot()

In [ ]:
session.mne.report.save("/Users/justin/Desktop/test.html", overwrite=True, open_browser=True)

In [ ]:
session.history()